In [1]:
import sys
print(sys.executable)

/home/onyxia/work/le-rail-suit-il-la-France-qui-grandit-/.venv/bin/python


# Exploration des contours (cartiflette)

Millésime 2026 disponible dans cartiflette (vérifié le 24/11/2026).
Fichiers bruts : communes (34 875), départements (101), régions (18), France entière, DOM compris.

In [2]:
import geopandas as gpd

#récupérer les données des communes de cartiflette
communes = gpd.read_parquet("../data/raw/contours_commune_2026.parquet")

print(communes.shape)
communes.head(1)

(34875, 22)


,ID,NOM,INSEE_COM,STATUT,POPULATION,AREA,CTCD,EPCI,EPT,ARR,...,UU2020,AAV2020,BV2022,INSEE_DEP,INSEE_REG,LIBELLE_DEPARTEMENT,LIBELLE_REGION,PAYS,SOURCE,geometry
0,COMMUNE_0000000000097101,Les Abymes,97101,Commune simple,51055,guadeloupe,971D,200018653,ZZZZZZZZZ,9712,...,9A701,9A1,97101,971,01,Guadeloupe,Guadeloupe,France,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-61.54898 16.25308, -61.5337 1..."


In [3]:
# permet de donner un résumé des variables que l'on a avec leur type et les potentielles données manquantes
resume = communes.dtypes.to_frame("type")
resume["manquants"] = communes.isna().sum()
resume

,type,manquants
ID,str,0
NOM,str,0
INSEE_COM,str,0
STATUT,str,0
POPULATION,int32,0
AREA,str,0
CTCD,str,0
EPCI,str,0
EPT,str,0
ARR,str,0


In [4]:
# 1. chaque code commune n'apparaît qu'une fois ?
print("codes en double :", communes["INSEE_COM"].duplicated().sum())

# 2. tous les codes font bien 5 caractères ?
print("longueurs des codes :", communes["INSEE_COM"].str.len().unique())

# 3. combien de communes dans les DOM (départements qui commencent par 97) ?
dom = communes["INSEE_DEP"].str.startswith("97")
print("communes des DOM :", dom.sum(), "| métropole :", (~dom).sum())

# 4. Paris, Lyon, Marseille : une seule commune ou des arrondissements ?
print(communes.loc[communes["NOM"].isin(["Paris", "Lyon", "Marseille"]), ["INSEE_COM", "NOM"]])

codes en double : 0
longueurs des codes : [5]
communes des DOM : 129 | métropole : 34746
      INSEE_COM        NOM
129       75056      Paris
32875     69123       Lyon
34150     13055  Marseille


## Constats

Contours comporte 34 875 communes, avec 22 colonnes, il n'y a pas de valeur manquante. `INSEE_COM` pourra peut-être être notre clé de jointure (?). 129 communes sont dans les DOM (département commençant par 97), on garde les 34 746 communes de métropole. Paris, Lyon et Marseille sont des communes à elles seules. 
On garde 5 colonnes: `INSEE_COM`, `NOM`, `INSEE_DEP`, `INSEE_REG`, `geometry`. La population et la surface viendront dans les séries historiques de l'INSEE. 

Suite à cela, lors du nettoyage, les colonnes seront renommées pour être standardisées.

In [5]:
import sys #librairie qui permet de calibrer python directement
sys.path.append("..")  # permet à python d'aller récupérer le dossier dans un autre onglet de code

from src.nettoyage import clean_contours

communes_propres = clean_contours(communes) # base de données clean
print(communes_propres.shape) # écrit la taille de la base de données
communes_propres.head(1) #vérification avec une variable

(34746, 5)


,code_commune,nom_commune,code_departement,code_region,geometry
0,75056,Paris,75,11,"POLYGON ((2.35198 48.90148, 2.32584 48.90095, ..."


In [6]:
# sauvegarde des contours propres
communes_propres.to_parquet("../data/clean/contours_communes.parquet")

# court bilan
print("lignes :", len(communes_propres)) #nombre d'observations
print("colonnes :", communes_propres.columns.tolist()) #nombres de variables
print("valeurs manquantes :", communes_propres.isna().sum().sum()) #s'assure que sur chaque colonne il n'y a pas de valeur manquante
print("projection : EPSG", communes_propres.crs.to_epsg()) #permet de controler les valeurs et les unités de nos variables

lignes : 34746
colonnes : ['code_commune', 'nom_commune', 'code_departement', 'code_region', 'geometry']
valeurs manquantes : 0
projection : EPSG 4326


## Bilan

Le fichier propre se situe à l'adresse : `data/clean/contours_communes.parquet`
- 34 746 communes de France métropolitaine (Corse comprise), millésime 2026 ;
- 5 colonnes : `code_commune`, `nom_commune`, `code_departement`, `code_region`, `geometry` ;
- aucune valeur manquante, code commune unique sur 5 caractères ;
- coordonnées en WGS84 (EPSG 4326), à convertir en Lambert 93 (EPSG 2154) pour les distances.